## Academic portfolio context

**Student / portfolio maintainer:** Mohammed Alaa Eddine GUERBOUZ  
**Institution:** INGETIS (Paris, France)  
**Programme:** Master-level studies · Generative AI / Large Language Models  
**Academic year:** 2026–2027  
**Topic:** Prompt Engineering with Gemini  
**Material type:** Course practical notebook / learning exercise  

**Problem explored:** How can prompt design and generation settings improve the usefulness, consistency and control of LLM responses?

> Portfolio annotation added in October 2026. Original instructional content and attribution are preserved below. This header identifies the student maintaining this study portfolio; it is **not** a claim that all classroom template code was originally authored by the student. The source filename was `Jour_1_Prompting_(Colab,_FR).ipynb`. See the folder README for requirements and limitations.


## Démarrer avec Google Colab

Ce notebook a été conçu à l'origine pour Kaggle ; cette version est adaptée pour **Google Colab**.

1. Ouvrez le fichier dans Colab : **Fichier > Importer le notebook** (ou déposez le fichier `.ipynb` dans Google Drive puis ouvrez-le avec Colab).
2. Enregistrez une copie dans votre Drive : **Fichier > Enregistrer une copie dans Drive**.

Chaque cellule de code dispose d'un bouton ▶️ **Exécuter** à sa gauche (ou utilisez `Maj + Entrée`). Ces notebooks sont prévus pour être exécutés dans l'ordre, de haut en bas, mais n'hésitez pas à ajouter de nouvelles cellules, à exécuter votre propre code et à explorer. Si vous êtes bloqué, utilisez **Exécution > Redémarrer la session** (ou **Déconnecter et supprimer l'environnement d'exécution**), puis relancez les cellules depuis le début.



## Démarrer avec l'API Gemini

Tous les exercices de ce notebook utilisent l'[API Gemini](https://ai.google.dev/gemini-api/) via le [SDK Python](https://pypi.org/project/google-genai/). Chacun de ces prompts est également accessible directement dans [Google AI Studio](https://aistudio.google.com/) : si vous préférez utiliser une interface web et vous passer du code pour cette activité, cherchez le lien <img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> AI Studio sous chaque prompt.

Ensuite, vous devez ajouter votre clé API dans Colab sous forme de **secret Colab** :

1. Dans la barre latérale gauche de Colab, cliquez sur l'icône **clé** 🔑 (« Secrets »).
2. Cliquez sur **Ajouter un secret**.
3. Dans **Nom**, saisissez exactement `GOOGLE_API_KEY` ; dans **Valeur**, collez votre clé API.
4. Activez l'interrupteur **Accès au notebook** pour ce secret.

Les secrets Colab sont liés à votre compte Google : ils ne sont pas enregistrés dans le notebook et ne sont pas partagés si vous partagez le fichier.

### Installer le SDK

In [ ]:
!pip install -U -q "google-genai"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


Importez le SDK ainsi que quelques utilitaires pour afficher les résultats.

In [ ]:
from google import genai
from google.genai import types

from IPython.display import HTML, Markdown, display

Mettez en place un mécanisme de nouvelle tentative (retry). Il vous permet de lancer « Tout exécuter » sans vous soucier du quota par minute.

In [ ]:
from google.api_core import retry


is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})

genai.models.Models.generate_content = retry.Retry(
    predicate=is_retriable)(genai.models.Models.generate_content)

### Configurer votre clé API

Pour exécuter la cellule suivante, votre clé API doit être enregistrée dans un [secret Colab](https://colab.research.google.com/notebooks/secrets.ipynb) nommé `GOOGLE_API_KEY` (voir les étapes ci-dessus).

Si vous n'avez pas encore de clé API, vous pouvez en obtenir une sur [AI Studio](https://aistudio.google.com/app/apikey). Vous trouverez des [instructions détaillées dans la documentation](https://ai.google.dev/gemini-api/docs/api-key).

La cellule ci-dessous lit d'abord le secret Colab. Par commodité, elle essaie aussi les secrets Kaggle puis la variable d'environnement `GOOGLE_API_KEY`, ce qui permet d'utiliser le même notebook ailleurs.

In [ ]:
import os

GOOGLE_API_KEY = None

# 1) Secrets Colab (icône clé dans la barre latérale gauche)
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass  # On n'est pas dans Colab

# 2) Secrets Kaggle (si le notebook est exécuté sur Kaggle)
if not GOOGLE_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    except Exception:
        pass

# 3) Variable d'environnement (exécution locale, VM, etc.)
if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError(
        "Clé API introuvable : ajoutez un secret Colab nommé GOOGLE_API_KEY "
        "et activez « Accès au notebook »."
    )

print("Clé API chargée ✅")

Clé API chargée ✅


Si vous obtenez une erreur du type `SecretNotFoundError` ou `NotebookAccessError`, vérifiez que le secret s'appelle exactement `GOOGLE_API_KEY` **et** que l'interrupteur **Accès au notebook** est activé dans le panneau Secrets (icône clé). Colab peut aussi afficher une fenêtre vous demandant d'autoriser l'accès : cliquez sur **Autoriser l'accès**.

### Lancer votre premier prompt

Dans cette étape, vous allez vérifier que votre clé API est bien configurée en envoyant une requête.

Le SDK Python utilise un [objet `Client`](https://googleapis.github.io/python-genai/genai.html#genai.client.Client) pour envoyer les requêtes à l'API. Le client permet de choisir le back-end utilisé (API Gemini ou Vertex AI) et gère l'authentification (la clé API).

**Remarque** : si une `TransportError` apparaît à cette étape, redémarrez la session une fois (**Exécution > Redémarrer la session**).

In [ ]:
# Modèle utilisé dans tout le notebook (gemini-2.0-flash a été arrêté le 1er juin 2026).
MODEL = "gemini-3.6-flash"

client = genai.Client(api_key=GOOGLE_API_KEY)

response = client.models.generate_content(
    model=MODEL,
    contents="Explain AI to me like I'm a kid.")

print(response.text)

RetryError: Timeout of 120.0s exceeded, last exception: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

La réponse est souvent renvoyée au format Markdown, que vous pouvez afficher directement dans ce notebook.

In [ ]:
Markdown(response.text)

### Démarrer une conversation

L'exemple précédent utilise une structure à un seul tour (texte en entrée, texte en sortie), mais vous pouvez aussi mettre en place une conversation à plusieurs tours.

In [ ]:
chat = client.chats.create(model=MODEL, history=[])
response = chat.send_message('Hello! My name is Zlork.')
print(response.text)

In [ ]:
response = chat.send_message('Can you tell me something interesting about dinosaurs?')
print(response.text)

Tant que l'objet `chat` existe, l'état de la conversation est conservé. Vérifiez-le en demandant au modèle s'il connaît le nom de l'utilisateur.

In [ ]:
response = chat.send_message('Do you remember what my name is?')
print(response.text)

### Choisir un modèle

L'API Gemini donne accès à plusieurs modèles de la famille Gemini. Découvrez les modèles disponibles et leurs capacités sur la [page de présentation des modèles](https://ai.google.dev/gemini-api/docs/models/gemini).

Dans cette étape, vous allez utiliser l'API pour lister tous les modèles disponibles.

In [ ]:
for model in client.models.list():
  print(model.name)

La réponse de [`models.list`](https://ai.google.dev/api/models#method:-models.list) renvoie aussi des informations supplémentaires sur les capacités de chaque modèle, comme les limites de jetons (tokens) et les paramètres pris en charge.

In [ ]:
from pprint import pprint

for model in client.models.list():
  if model.name == f'models/{MODEL}':
    pprint(model.to_json_dict())
    break

## Explorer les paramètres de génération



### Longueur de la sortie

Lorsque vous générez du texte avec un LLM, la longueur de la sortie influe sur le coût et les performances. Générer plus de jetons demande plus de calcul, ce qui augmente la consommation d'énergie, la latence et le coût.

Pour empêcher le modèle de générer des jetons au-delà d'une limite, vous pouvez indiquer le paramètre `max_output_tokens` lors de l'appel à l'API Gemini. Ce paramètre n'influence pas la manière dont les jetons sont générés : la sortie ne devient pas plus concise sur le plan du style ou du contenu, la génération s'arrête simplement une fois la longueur atteinte. Un travail sur le prompt peut être nécessaire pour obtenir une réponse complète dans la limite choisie.

> **Remarque (modèles Gemini 3)** : ces modèles « réfléchissent » avant de répondre, et les jetons de réflexion sont comptés dans `max_output_tokens`. Avec une limite très basse, la réponse peut donc être tronquée, voire vide (`None`). Dans ce cas, augmentez la limite.

In [ ]:
from google.genai import types

short_config = types.GenerateContentConfig(max_output_tokens=200)

response = client.models.generate_content(
    model=MODEL,
    config=short_config,
    contents='Write a 1000 word essay on the importance of olives in modern society.')

print(response.text)

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    config=short_config,
    contents='Write a short poem on the importance of olives in modern society.')

print(response.text)

Testez avec vos propres prompts. Essayez un prompt avec une limite de sortie restrictive, puis adaptez le prompt pour qu'il fonctionne dans cette limite.

### Température

La température contrôle le degré d'aléatoire dans la sélection des jetons. Une température élevée augmente le nombre de jetons candidats parmi lesquels le prochain jeton est choisi, et peut produire des résultats plus variés. Une température basse a l'effet inverse : une température de 0 correspond à un décodage glouton (greedy decoding), qui choisit le jeton le plus probable à chaque étape.

La température ne garantit pas le caractère aléatoire, mais elle permet d'« orienter » un peu la sortie.

In [ ]:
high_temp_config = types.GenerateContentConfig(temperature=2.0)


for _ in range(5):
  response = client.models.generate_content(
      model=MODEL,
      config=high_temp_config,
      contents='Pick a random colour... (respond in a single word)')

  if response.text:
    print(response.text, '-' * 25)

Essayez maintenant le même prompt avec une température de zéro. La sortie n'est pas totalement déterministe, car d'autres paramètres influencent la sélection des jetons, mais les résultats auront tendance à être plus stables.

In [ ]:
low_temp_config = types.GenerateContentConfig(temperature=0.0)

for _ in range(5):
  response = client.models.generate_content(
      model=MODEL,
      config=low_temp_config,
      contents='Pick a random colour... (respond in a single word)')

  if response.text:
    print(response.text, '-' * 25)

### Top-P

Comme la température, le paramètre top-P sert à contrôler la diversité de la sortie du modèle.

Top-P définit un seuil de probabilité : les jetons sont ajoutés aux candidats, du plus probable au moins probable, jusqu'à ce que leur probabilité cumulée dépasse ce seuil. Un top-P de 0 équivaut généralement à un décodage glouton, et un top-P de 1 retient en général tous les jetons du vocabulaire du modèle.

Vous rencontrerez peut-être aussi le top-K dans la littérature sur les LLM. Le top-K n'était pas configurable dans la série Gemini 2.0, mais pouvait être modifié dans des modèles plus anciens. Le top-K est un entier positif qui définit le nombre de jetons les plus probables parmi lesquels choisir le jeton de sortie. Un top-K de 1 sélectionne un seul jeton, ce qui revient à un décodage glouton.


Exécutez cet exemple plusieurs fois, modifiez les réglages et observez l'effet sur la sortie.

In [ ]:
model_config = types.GenerateContentConfig(
    # Valeurs par défaut de gemini-2.0-flash (modèle d'origine du cours).
    temperature=1.0,
    top_p=0.95,
)

story_prompt = "You are a creative writer. Write a short story about a cat who goes on an adventure."
response = client.models.generate_content(
    model=MODEL,
    config=model_config,
    contents=story_prompt)

print(response.text)

## Prompting

Cette section reprend plusieurs prompts du chapitre pour que vous puissiez les tester directement avec l'API. Modifiez le texte pour voir comment chaque prompt se comporte avec des instructions différentes, davantage d'exemples ou tout autre changement qui vous vient à l'esprit.

### Zero-shot

Les prompts zero-shot décrivent directement la demande au modèle, sans exemple.

<table align=left>
  <td>
    <a target="_blank" href="https://aistudio.google.com/prompts/1gzKKgDHwkAvexG5Up0LMtl1-6jKMKe4g"><img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> Ouvrir dans AI Studio</a>
  </td>
</table>

In [ ]:
model_config = types.GenerateContentConfig(
    temperature=0.1,
    top_p=1,
    # Avec les modèles Gemini 3, les jetons de réflexion comptent dans cette limite :
    # si la réponse est vide (None), augmentez-la (par exemple à 1024).
    max_output_tokens=5,
)

zero_shot_prompt = """Classify movie reviews as POSITIVE, NEUTRAL or NEGATIVE.
Review: "Her" is a disturbing study revealing the direction
humanity is headed if AI is allowed to keep evolving,
unchecked. I wish there were more movies like this masterpiece.
Sentiment: """

response = client.models.generate_content(
    model=MODEL,
    config=model_config,
    contents=zero_shot_prompt)

print(response.text)

#### Mode enum

Les modèles sont entraînés à générer du texte, et même si les modèles Gemini suivent très bien les instructions, d'autres modèles produisent parfois plus de texte que souhaité. Dans l'exemple précédent, le modèle renvoie le label, mais il peut parfois le faire précéder de « Sentiment », et sans limite de jetons, il peut aussi ajouter une explication. Voir [ce prompt dans AI Studio](https://aistudio.google.com/prompts/1gzKKgDHwkAvexG5Up0LMtl1-6jKMKe4g) pour un exemple.

L'API Gemini propose un [mode enum](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Enum.ipynb) qui permet de contraindre la sortie à un ensemble fixe de valeurs.

In [ ]:
import enum

class Sentiment(enum.Enum):
    POSITIVE = "positive"
    NEUTRAL = "neutral"
    NEGATIVE = "negative"


response = client.models.generate_content(
    model=MODEL,
    config=types.GenerateContentConfig(
        response_mime_type="text/x.enum",
        response_schema=Sentiment
    ),
    contents=zero_shot_prompt)

print(response.text)

Lorsque vous utilisez une sortie contrainte comme un enum, le SDK Python tente de convertir automatiquement la réponse texte du modèle en objet Python. Le résultat est stocké dans le champ `response.parsed`.

In [ ]:
enum_response = response.parsed
print(enum_response)
print(type(enum_response))

### One-shot et few-shot

Fournir un exemple de la réponse attendue s'appelle un prompt « one-shot ». Lorsque vous fournissez plusieurs exemples, on parle de prompt « few-shot ».

<table align=left>
  <td>
    <a target="_blank" href="https://aistudio.google.com/prompts/1jjWkjUSoMXmLvMJ7IzADr_GxHPJVV2bg"><img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> Ouvrir dans AI Studio</a>
  </td>
</table>


In [ ]:
few_shot_prompt = """Parse a customer's pizza order into valid JSON:

EXAMPLE:
I want a small pizza with cheese, tomato sauce, and pepperoni.
JSON Response:
```
{
"size": "small",
"type": "normal",
"ingredients": ["cheese", "tomato sauce", "pepperoni"]
}
```

EXAMPLE:
Can I get a large pizza with tomato sauce, basil and mozzarella
JSON Response:
```
{
"size": "large",
"type": "normal",
"ingredients": ["tomato sauce", "basil", "mozzarella"]
}
```

ORDER:
"""

customer_order = "Give me a large with cheese & pineapple"

response = client.models.generate_content(
    model=MODEL,
    config=types.GenerateContentConfig(
        temperature=0.1,
        top_p=1,
        max_output_tokens=250,
    ),
    contents=[few_shot_prompt, customer_order])

print(response.text)

#### Mode JSON

Pour contrôler le schéma et vous assurer de ne recevoir que du JSON (sans autre texte ni Markdown), vous pouvez utiliser le [mode JSON](https://github.com/google-gemini/cookbook/blob/main/quickstarts/JSON_mode.ipynb) de l'API Gemini. Il contraint le décodage du modèle : la sélection des jetons est guidée par le schéma fourni.

In [ ]:
import typing_extensions as typing

class PizzaOrder(typing.TypedDict):
    size: str
    ingredients: list[str]
    type: str


response = client.models.generate_content(
    model=MODEL,
    config=types.GenerateContentConfig(
        temperature=0.1,
        response_mime_type="application/json",
        response_schema=PizzaOrder,
    ),
    contents="Can I have a large dessert pizza with apple and chocolate")

print(response.text)

### Chaîne de pensée (Chain of Thought, CoT)

Interroger directement un LLM peut donner des réponses rapides et économes (en jetons de sortie), mais sujettes aux hallucinations. La réponse peut « sembler » correcte (sur le plan de la langue et de la syntaxe) tout en étant fausse sur le fond et le raisonnement.

Le prompting par chaîne de pensée consiste à demander au modèle de produire les étapes intermédiaires de son raisonnement, ce qui donne généralement de meilleurs résultats, surtout combiné à des exemples few-shot. Notez que cette technique n'élimine pas complètement les hallucinations et qu'elle coûte en général plus cher, du fait du nombre accru de jetons.

Les modèles comme ceux de la famille Gemini sont entraînés à être « bavards » ou « réfléchis » et fournissent des étapes de raisonnement sans qu'on le leur demande. Pour cet exemple simple, vous pouvez donc demander au modèle d'être plus direct dans le prompt afin de forcer une réponse sans raisonnement. Relancez cette étape si le modèle a de la chance et trouve la bonne réponse du premier coup.

> **Remarque** : les modèles Gemini 3 raisonnent en interne avant de répondre ; ils trouvent donc souvent la bonne réponse même sans « étape par étape ».

In [ ]:
prompt = """When I was 4 years old, my partner was 3 times my age. Now, I
am 20 years old. How old is my partner? Return the answer directly."""

response = client.models.generate_content(
    model=MODEL,
    contents=prompt)

print(response.text)

Essayez maintenant la même approche, mais en indiquant au modèle qu'il doit « réfléchir étape par étape » (« think step by step »).

In [ ]:
prompt = """When I was 4 years old, my partner was 3 times my age. Now,
I am 20 years old. How old is my partner? Let's think step by step."""

response = client.models.generate_content(
    model=MODEL,
    contents=prompt)

Markdown(response.text)

### ReAct : raisonner et agir

Dans cet exemple, vous allez exécuter un prompt ReAct directement avec l'API Gemini et effectuer vous-même les étapes de recherche. Comme ce prompt suit une structure bien définie, il existe des frameworks qui l'encapsulent dans des API plus simples à utiliser et qui effectuent automatiquement les appels d'outils, comme l'exemple LangChain du livre blanc « Prompting ».

Pour essayer avec le moteur de recherche de Wikipédia, consultez l'exemple du cookbook [Searching Wikipedia with ReAct](https://github.com/google-gemini/cookbook/blob/main/examples/Search_Wikipedia_using_ReAct.ipynb).


> Remarque : le prompt et les exemples en contexte utilisés ici proviennent de [https://github.com/ysymyth/ReAct](https://github.com/ysymyth/ReAct), publié sous [licence MIT](https://opensource.org/licenses/MIT), Copyright (c) 2023 Shunyu Yao.

<table align=left>
  <td>
    <a target="_blank" href="https://aistudio.google.com/prompts/18oo63Lwosd-bQ6Ay51uGogB3Wk3H8XMO"><img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> Ouvrir dans AI Studio</a>
  </td>
</table>


In [ ]:
model_instructions = """
Solve a question answering task with interleaving Thought, Action, Observation steps. Thought can reason about the current situation,
Observation is understanding relevant information from an Action's output and Action can be one of three types:
 (1) <search>entity</search>, which searches the exact entity on Wikipedia and returns the first paragraph if it exists. If not, it
     will return some similar entities to search and you can try to search the information from those topics.
 (2) <lookup>keyword</lookup>, which returns the next sentence containing keyword in the current context. This only does exact matches,
     so keep your searches short.
 (3) <finish>answer</finish>, which returns the answer and finishes the task.
"""

example1 = """Question
Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?

Thought 1
The question simplifies to "The Simpsons" character Milhouse is named after who. I only need to search Milhouse and find who it is named after.

Action 1
<search>Milhouse</search>

Observation 1
Milhouse Mussolini Van Houten is a recurring character in the Fox animated television series The Simpsons voiced by Pamela Hayden and created by Matt Groening.

Thought 2
The paragraph does not tell who Milhouse is named after, maybe I can look up "named after".

Action 2
<lookup>named after</lookup>

Observation 2
Milhouse was named after U.S. president Richard Nixon, whose middle name was Milhous.

Thought 3
Milhouse was named after U.S. president Richard Nixon, so the answer is Richard Nixon.

Action 3
<finish>Richard Nixon</finish>
"""

example2 = """Question
What is the elevation range for the area that the eastern sector of the Colorado orogeny extends into?

Thought 1
I need to search Colorado orogeny, find the area that the eastern sector of the Colorado orogeny extends into, then find the elevation range of the area.

Action 1
<search>Colorado orogeny</search>

Observation 1
The Colorado orogeny was an episode of mountain building (an orogeny) in Colorado and surrounding areas.

Thought 2
It does not mention the eastern sector. So I need to look up eastern sector.

Action 2
<lookup>eastern sector</lookup>

Observation 2
The eastern sector extends into the High Plains and is called the Central Plains orogeny.

Thought 3
The eastern sector of Colorado orogeny extends into the High Plains. So I need to search High Plains and find its elevation range.

Action 3
<search>High Plains</search>

Observation 3
High Plains refers to one of two distinct land regions

Thought 4
I need to instead search High Plains (United States).

Action 4
<search>High Plains (United States)</search>

Observation 4
The High Plains are a subregion of the Great Plains. From east to west, the High Plains rise in elevation from around 1,800 to 7,000 ft (550 to 2,130m).

Thought 5
High Plains rise in elevation from around 1,800 to 7,000 ft, so the answer is 1,800 to 7,000 ft.

Action 5
<finish>1,800 to 7,000 ft</finish>
"""

# Inventez d'autres exemples vous-même, ou parcourez https://github.com/ysymyth/ReAct/

Pour obtenir une seule étape à la fois, en ignorant les éventuelles étapes « Observation » hallucinées, vous allez utiliser `stop_sequences` pour interrompre la génération. Les étapes sont, dans l'ordre : `Thought` (pensée), `Action` et `Observation`.

In [ ]:
question = """Question
Who was the youngest author listed on the transformers NLP paper?
"""

# C'est vous qui effectuez l'Action : on génère donc jusqu'à l'Observation, sans l'inclure.
react_config = types.GenerateContentConfig(
    stop_sequences=["\nObservation"],
    system_instruction=model_instructions + example1 + example2,
)

# Crée une conversation dans laquelle les instructions et les exemples sont préchargés.
react_chat = client.chats.create(
    model=MODEL,
    config=react_config,
)

resp = react_chat.send_message(question)
print(resp.text)

Vous pouvez maintenant faire cette recherche vous-même et fournir le résultat au modèle.

In [ ]:
observation = """Observation 1
[1706.03762] Attention Is All You Need
Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.
"""
resp = react_chat.send_message(observation)
print(resp.text)

Ce processus se répète jusqu'à ce que l'action `<finish>` soit atteinte. Vous pouvez continuer vous-même si vous le souhaitez, ou essayer l'[exemple Wikipédia](https://github.com/google-gemini/cookbook/blob/main/examples/Search_Wikipedia_using_ReAct.ipynb) pour voir un système ReAct entièrement automatisé à l'œuvre.

## Mode réflexion (thinking mode)

Le modèle expérimental Gemini Flash 2.0 « Thinking » a été entraîné à générer le « processus de réflexion » qu'il suit dans le cadre de sa réponse. Le modèle Flash Thinking offre ainsi de meilleures capacités de raisonnement.

Utiliser un modèle en « mode réflexion » permet d'obtenir des réponses de grande qualité sans prompting spécialisé comme dans les approches précédentes. Si cette technique est efficace, c'est notamment parce qu'elle amène le modèle à générer des informations pertinentes (« brainstorming » ou « pensées ») qui servent ensuite de contexte pour produire la réponse finale.

Notez qu'avec l'API, vous obtenez la réponse finale du modèle, mais les pensées ne sont pas renvoyées. Pour voir les pensées intermédiaires, essayez [le modèle en mode réflexion dans AI Studio](https://aistudio.google.com/prompts/new_chat?model=gemini-2.0-flash-thinking-exp-01-21).

> **Mise à jour** : le modèle `gemini-2.0-flash-thinking-exp` n'existe plus. Les modèles Gemini 3 intègrent la réflexion par défaut : la cellule ci-dessous utilise donc simplement `MODEL`.

<table align=left>
  <td>
    <a target="_blank" href="https://aistudio.google.com/prompts/1Z991SV7lZZZqioOiqIUPv9a9ix-ws4zk"><img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> Ouvrir dans AI Studio</a>
  </td>
</table>

In [ ]:
import io
from IPython.display import Markdown, clear_output


response = client.models.generate_content_stream(
    model=MODEL,  # à l'origine : 'gemini-2.0-flash-thinking-exp' (modèle retiré)
    contents='Who was the youngest author listed on the transformers NLP paper?',
)

buf = io.StringIO()
for chunk in response:
    if chunk.text:  # certains fragments du flux peuvent ne pas contenir de texte
        buf.write(chunk.text)
        # Affiche la réponse au fur et à mesure du streaming
        print(chunk.text, end='')

# Puis affiche la réponse complète au format Markdown.
clear_output()
Markdown(buf.getvalue())

## Prompting pour le code

### Générer du code

Les modèles de la famille Gemini peuvent générer du code, des fichiers de configuration et des scripts. C'est utile pour apprendre à programmer, découvrir un nouveau langage ou produire rapidement un premier jet.

Comme les LLM peuvent se tromper et reproduire des données d'entraînement, il est indispensable de lire et de tester le code avant de l'utiliser, et de respecter les licences applicables.

<table align=left>
  <td>
    <a target="_blank" href="https://aistudio.google.com/prompts/1YX71JGtzDjXQkgdes8bP6i3oH5lCRKxv"><img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> Ouvrir dans AI Studio</a>
  </td>
</table>

In [ ]:
# Les modèles Gemini aiment parler : si vous ne voulez que du code,
# mieux vaut leur préciser de s'y limiter.
code_prompt = """
Write a Python function to calculate the factorial of a number. No explanation, provide only the code.
"""

response = client.models.generate_content(
    model=MODEL,
    config=types.GenerateContentConfig(
        temperature=1,
        top_p=1,
        max_output_tokens=1024,
    ),
    contents=code_prompt)

Markdown(response.text)

### Exécution de code

L'API Gemini peut aussi exécuter automatiquement le code généré et renvoyer le résultat.

<table align=left>
  <td>
    <a target="_blank" href="https://aistudio.google.com/prompts/11veFr_VYEwBWcLkhNLr-maCG0G8sS_7Z"><img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> Ouvrir dans AI Studio</a>
  </td>
</table>

In [ ]:
from pprint import pprint

config = types.GenerateContentConfig(
    tools=[types.Tool(code_execution=types.ToolCodeExecution())],
)

code_exec_prompt = """
Generate the first 14 odd prime numbers, then calculate their sum.
"""

response = client.models.generate_content(
    model=MODEL,
    config=config,
    contents=code_exec_prompt)

for part in response.candidates[0].content.parts:
  pprint(part.to_json_dict())
  print("-----")

Cette réponse contient plusieurs parties : une partie texte d'ouverture et une de clôture, qui correspondent à des réponses classiques, une partie `executable_code` qui contient le code généré, et une partie `code_execution_result` qui contient le résultat de son exécution.

Vous pouvez les explorer une par une.

In [ ]:
for part in response.candidates[0].content.parts:
    if part.text:
        display(Markdown(part.text))
    elif part.executable_code:
        display(Markdown(f'```python\n{part.executable_code.code}\n```'))
    elif part.code_execution_result:
        if part.code_execution_result.outcome != 'OUTCOME_OK':
            display(Markdown(f'## Status {part.code_execution_result.outcome}'))

        display(Markdown(f'```\n{part.code_execution_result.output}\n```'))

### Expliquer du code

Les modèles de la famille Gemini peuvent aussi vous expliquer du code. Dans cet exemple, vous leur transmettez un [script bash](https://github.com/magicmonty/bash-git-prompt) et posez quelques questions.

<table align=left>
  <td>
    <a target="_blank" href="https://aistudio.google.com/prompts/1N7LGzWzCYieyOf_7bAG4plrmkpDNmUyb"><img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> Ouvrir dans AI Studio</a>
  </td>
</table>

In [ ]:
file_contents = !curl https://raw.githubusercontent.com/magicmonty/bash-git-prompt/refs/heads/master/gitprompt.sh

explain_prompt = f"""
Please explain what this file does at a very high level. What is it, and why would I use it?

```
{file_contents}
```
"""

response = client.models.generate_content(
    model=MODEL,
    contents=explain_prompt)

Markdown(response.text)

## Pour aller plus loin

Pour approfondir le prompting :

* Lisez l'[introduction au prompting](https://ai.google.dev/gemini-api/docs/prompting-intro) dans la documentation de l'API Gemini ;
* Explorez la [galerie de prompts](https://ai.google.dev/gemini-api/prompts) de l'API Gemini et testez-les dans AI Studio ;
* Parcourez le cookbook de l'API Gemini pour des [exemples inspirants](https://github.com/google-gemini/cookbook/blob/main/examples/) et des [guides de démarrage pédagogiques](https://github.com/google-gemini/cookbook/blob/main/quickstarts/).
